# Load Trained Model and Create Predictions
This notebook loads a previously trained model and uses it to create predictions on new or held-out data. It documents the full prediction workflow so results are reproducible and easy to inspect.


## Setup

In [ ]:
# Add project root to Python path
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Standard library imports
import os
import yaml
import pickle
import pandas as pd

# Local imports from src package
from src.data.data_module import Ice_Cover_Dataset

In [ ]:
# Load the Model and Config along with any other necessary parameters / constants

# Define the run to use, confirm it exists
run_name = "LIF_DL_Best"
run_dir = f"../results/{run_name}"
run_exists = os.path.exists(run_dir)
if not run_exists:
    raise FileNotFoundError(f"Run directory {run_dir} does not exist.")

# Check that the checkpoint file exists
ckpt_name = "lif-dl-epoch=034-val_loss=0.303.ckpt"
if not os.path.exists(os.path.join(run_dir, ckpt_name)):
    raise FileNotFoundError(f"Checkpoint file {ckpt_name} does not exist in {run_dir}.")

# Load the config file
config_path = run_dir + "/config.yaml"
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config file {config_path} does not exist.")
else:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

# Check that data directories exist
data_dir = "../data/nc/"
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"Data directory {data_dir} does not exist.")

# Load the dataset indices and stats
dataset_stats_path = run_dir + "/dataset_stats.pkl"
if not os.path.exists(dataset_stats_path):
    raise FileNotFoundError(f"Dataset stats file {dataset_stats_path} does not exist.")
else:
    with open(dataset_stats_path, 'rb') as f:
        dataset_stats = pickle.load(f)

In [ ]:
import xarray as xr
ds = xr.open_dataset("../data/nc/great_bear_lake.nc")
ds

In [ ]:
# 1. Load the testing dataset to use for predictions
start_date = "2018-01-01"
end_date = "2021-12-17"
sequence_length = config['sequence_length']
window_size = sequence_length * 2  # Historical + Prediction window
test_dates = pd.date_range(start_date, end_date)
sites = [
    "great_slave_lake",
    "great_bear_lake",
    "lake_athabasca",
    "lake_winnipeg",
    "reindeer_lake",
]
variables = config['variables']

# Load the testing dataset
test_data = Ice_Cover_Dataset(data_dir, sites, start_date, end_date, sequence_length, variables)
print(len(test_data), "testing sequences loaded.")

# Grab a sample from the test dataset to verify everything is working
sample_idx = 0
sample_input, lake_mask, sample_target = test_data[sample_idx]
print("Sample input shape:", sample_input.shape)
print("Sample target shape:", sample_target.shape)

print("Available variables:", variables)

In [ ]:
# 2. Load the trained model from checkpoint
import torch
from src.model.lif_dl import LIF_DL
from src.model.model import LitModel, create_model

# Instantiate the LIF_DL model using the config
model = create_model(config)

# Load the model weights from the checkpoint
ckpt_path = os.path.join(run_dir, ckpt_name)
lit_model = LitModel.load_from_checkpoint(ckpt_path, model=model, config=config)

# Set the model to evaluation mode
lit_model.eval()
lit_model.to(torch.device('cpu'))

# Print model summary
print(lit_model)

In [ ]:
# Test passing the sample input through the model
with torch.no_grad():
    sample_input = sample_input.unsqueeze(0)  # Add batch dimension
    sample_output = lit_model(sample_input)
    print("Sample output shape:", sample_output.shape)

## Experimenting with the Trained Model

### 1. Exploring the Testing Data

In [ ]:
"""
This cell will plot some sample input and targets, to help you visualize what the model is seeing.

The plot will be organized as follows:
Time: y axis
Variables: x axis

Where the user specifies 3 variables to plot. In addition to those three, there is one column for "Past ice cover" (an input) and one for target ice cover.
Therefore there will be 7 rows and 5 columns in total.

The one hot ice cover will be converted back to categorical for plotting.
"""

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap, BoundaryNorm

## ===== Prepare Data for Plotting ===== ##
## CHANGE THESE PARAMETERS TO PLOT DIFFERENT VARIABLES / DATES ##
# Specify which variable to plot
# test_data.scale_data = False # Disable scaling for plotting
variable_to_plot = 'lake_depth'
variable_index = test_data.vars.index(variable_to_plot)
print(f"Plotting variable: {variable_to_plot} (index {variable_index})")

# Specify the date for plotting, which defines the first date in the input sequence
plot_date = "2018-05-25"  # First date in the test set
plot_lake = "great_slave_lake"
plot_idx = test_data.get_lake_date_index(plot_date, plot_lake)

# Get the input, mask, and target for the specified date
input_data, lake_mask, target_data = test_data[plot_idx]
input_data = input_data.numpy()
target_data = target_data.numpy()
lake_mask = lake_mask.numpy()[0,0]
print(target_data.shape)

# Extract the variables of interest from the input data
input_var = input_data[:, variable_index, :, :]
past_ice_cover = input_data[:, -3:, :, :]  # Assuming last 3 channels are one-hot ice cover

# Convert the one-hot ice cover to categorical
past_ice_cover_cat = np.argmax(past_ice_cover, axis=1)
target_ice_cover_cat = np.argmax(target_data, axis=1)
print(target_ice_cover_cat.shape)

# Where lake_mask is False, set the past and target ice cover to -1 (indicating no lake)
past_ice_cover_cat = np.where(lake_mask, past_ice_cover_cat, -1)
target_ice_cover_cat = np.where(lake_mask, target_ice_cover_cat, -1)

## ===== Plotting ===== ##
# define a colour map for the ice cover, using discrete colours (grey, blue, white)
cmap = ListedColormap(['darkgrey', 'blue', 'white'])
norm = BoundaryNorm([-1, 0, 1, 2], cmap.N)
fig, axes = plt.subplots(nrows=3, ncols=7, figsize=(20,8), sharex=True)
time_steps = input_data.shape[0]
x = np.arange(input_data.shape[2])  # Spatial dimension

date_window = pd.date_range(plot_date, periods=window_size)

for t in range(time_steps):
    
    # Plot past ice cover
    im = axes[0, t].imshow(past_ice_cover_cat[t, :], aspect='auto', cmap=cmap, norm=norm, origin='lower')
    axes[0, t].set_title(date_window[t].strftime("%Y-%m-%d"), fontsize=10)
    
    im_var = axes[1, t].imshow(input_var[t, :, :], aspect='auto', origin='lower', cmap='cividis', vmin=np.min(input_var), vmax=np.max(input_var))
    # Add date label above the axis
    date_label = date_window[-7+t].strftime("%Y-%m-%d")
    axes[1, t].set_title(date_label, fontsize=10)
    
    # Plot target ice cover
    im = axes[2, t].imshow(target_ice_cover_cat[t, :], aspect='auto', cmap=cmap, norm=norm, origin='lower')
    axes[2, t].set_title(date_label, fontsize=10)

# Add labels to the left side
axes[1, 0].set_ylabel(f"{variable_to_plot}", fontsize=12)
axes[0, 0].set_ylabel("Past Ice Cover", fontsize=12)
axes[2, 0].set_ylabel("Target Ice Cover", fontsize=12)

# Add colorbars
cbar = fig.colorbar(im, ax=axes[0:1, :], orientation='vertical', fraction=0.02, pad=0.01)
cbar.set_ticks([-0.5, 0.5, 1.5])
cbar.set_ticklabels(['No Lake', 'Water', 'Ice'])

cbar = fig.colorbar(im_var, ax=axes[1:2, :], orientation='vertical', fraction=0.02, pad=0.01, cmap='cividis')

cbar = fig.colorbar(im, ax=axes[2:3, :], orientation='vertical', fraction=0.02, pad=0.01)
cbar.set_ticks([-0.5, 0.5, 1.5])
cbar.set_ticklabels(['No Lake', 'Water', 'Ice'])

### 2. Make Model Predictions and Plot the Results

**Note on Reproducibility:** The model contains dropout layers that are active during training but must be disabled during inference. The prediction cell below explicitly calls `lit_model.eval()` and sets random seeds to ensure completely reproducible predictions. Always ensure the model is in evaluation mode before making predictions!

In [ ]:
"""
Now we will make predictions with the model on the test dataset and plot the results similarly to above.
"""

# Scale the data using the training set statistics
test_data.toggle_scaling(dataset_stats)

# Specify the date and lake site to predict
plot_date = pd.to_datetime("2018-05-28")  # First date in the sequence of targets
plot_lake = "great_slave_lake"
plot_idx = test_data.get_lake_date_index(plot_date, plot_lake)
print("Plotting predictions for", plot_lake, "on", plot_date, "(index", plot_idx, ")")

# Get the input, mask, and target for the specified date
input_data, lake_mask, target_data = test_data[plot_idx]

# IMPORTANT: Set model to evaluation mode to disable dropout and ensure reproducibility
lit_model.eval()

# Set random seed for complete reproducibility (in case of any stochastic operations)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Make prediction with the model
with torch.no_grad():
    model_output = lit_model(input_data.unsqueeze(0))  # Add batch dimension
    print("Model output shape:", model_output.shape)
    
# Prepare data for plotting
model_output = model_output.squeeze().numpy()  # Remove batch dimension
lake_mask = lake_mask.numpy()[0,0]
target_data = target_data.numpy()
input_ice_cover = input_data.numpy()[:, -3:, :, :]

# Convert the sigmoid predicted ice cover to categorical
predicted_ice_cover_cat = model_output.round().astype(int)
target_ice_cover_cat = np.argmax(target_data, axis=1)
input_ice_cover_cat = np.argmax(input_ice_cover, axis=1)

# Apply the lake mask
predicted_ice_cover_cat = np.where(lake_mask, predicted_ice_cover_cat, -1)
target_ice_cover_cat = np.where(lake_mask, target_ice_cover_cat, -1)
input_ice_cover_cat = np.where(lake_mask, input_ice_cover_cat, -1)

## ===== Plotting Predictions ===== ##
# define a colour map for the ice cover, using discrete colours (grey, blue, white)
cmap = ListedColormap(['darkgrey', 'blue', 'white'])
norm = BoundaryNorm([-1, 0, 1, 2], cmap.N)
fig, axes = plt.subplots(nrows=3, ncols=7, figsize=(20,6), sharex=True)
time_steps = input_data.shape[0]
x = np.arange(input_data.shape[3])  # Spatial dimension
date_window = pd.date_range(plot_date-pd.Timedelta(days=sequence_length),
                            periods=window_size)

for t in range(time_steps):
    
    # Plot past ice cover
    im = axes[0, t].imshow(input_ice_cover_cat[t], aspect='auto', cmap=cmap, norm=norm, origin='lower')
    axes[0, t].set_title(date_window[t].strftime("%Y-%m-%d"), fontsize=10)
    
    # Plot variable (not available here, so we skip this)
    
    # Plot predicted ice cover
    im = axes[1, t].imshow(predicted_ice_cover_cat[t], aspect='auto', cmap=cmap, norm=norm, origin='lower')
    axes[1, t].set_title(date_window[-7+t].strftime("%Y-%m-%d"), fontsize=10)
    
    # Plot target ice cover
    im = axes[2, t].imshow(target_ice_cover_cat[t], aspect='auto', cmap=cmap, norm=norm, origin='lower')
    axes[2, t].set_title(date_window[-7+t].strftime("%Y-%m-%d"), fontsize=10)

# Add labels to the left side
axes[1, 0].set_ylabel("Predicted Ice Cover", fontsize=12)
axes[0, 0].set_ylabel("Past Ice Cover", fontsize=12)
axes[2, 0].set_ylabel("Target Ice Cover", fontsize=12)

# Autoregressive Forecasting

In [ ]:
from src.model.predictor import Predictor
import pandas as pd

config["run_dir"] = run_dir
predictor = Predictor(config)

site = "great_bear_lake"
start_date = "2018-05-01"
end_date = "2018-06-30"

model_pred, targets = predictor.forecast(test_data, site, start_date, end_date)

print("model_pred shape:", model_pred.shape)
print("targets shape:", targets.shape)

In [ ]:
# Try plotting the predictions vs targets as before
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
# define a colour map for the ice cover, using discrete colours (grey, blue, white)
cmap = ListedColormap(['darkgrey', 'blue', 'white'])
norm = BoundaryNorm([-1, 0, 1, 2], cmap.N)
fig, axes = plt.subplots(nrows=2, ncols=7, figsize=(20,6), sharex=True)
time_steps = 7
offset = 45 # Adjust this to plot different weeks
x = np.arange(model_pred.shape[2])  # Spatial dimension
date_window = pd.date_range(start=start_date, periods=len(model_pred))

for t in range(time_steps):
    
    # Plot predicted ice cover
    im = axes[0, t].imshow(model_pred[t+offset], aspect='auto', cmap=cmap, norm=norm, origin='lower')
    axes[0, t].set_title(date_window[t+offset].strftime("%Y-%m-%d"), fontsize=10)
    
    # Plot target ice cover
    im = axes[1, t].imshow(targets[t+offset], aspect='auto', cmap=cmap, norm=norm, origin='lower')
    axes[1, t].set_title(date_window[t+offset].strftime("%Y-%m-%d"), fontsize=10)



## Visualize Forecasting Results
requires a run of forecast.py to have completed successfully!

In [ ]:
site = "great_bear_lake"

# Load the forecasting data:
forecast_path = "../results/LIF_DL_Best/forecasts/"
ds_pred = xr.open_dataset(f"{forecast_path}{site}_forecast.nc")
# print(ds_pred)

test_start = config['test_start']
test_end = config['test_end']
test_dates = pd.date_range(test_start, test_end)
print(len(test_dates))  # Number of days in the test set

# The test dates align with the first dimension of the array.
# Load the testing data from xarray storage
ds = xr.open_dataset(f"../data/nc/{site}.nc")
ims_data = ds.IMS_Surface_Values
lake_mask = ds.lake_mask.values

# Plot predictions against targets for a specific date range
start_date = "2021-06-30"
date_window = pd.date_range(start=start_date, periods=7)

fig, axes = plt.subplots(nrows=2, ncols=7, figsize=(20,6), sharex=True)
time_steps = 7

for t in range(time_steps):
    
    # Plot predicted ice cover
    pred = ds_pred.ice_cover_prediction.sel(time=date_window[t]).values
    im = axes[0, t].imshow(pred, aspect='auto', cmap=cmap, norm=norm, origin='lower')
    axes[0, t].set_title(date_window[t].strftime("%Y-%m-%d"), fontsize=10)
    
    # Plot target ice cover
    target = ims_data.sel(time=date_window[t]).values
    # Apply lake mask if necessary
    target = np.where(lake_mask, target, -1)
    im = axes[1, t].imshow(target, aspect='auto', cmap=cmap, norm=norm, origin='lower')
    axes[1, t].set_title(date_window[t].strftime("%Y-%m-%d"), fontsize=10)

# Add labels to the left side
axes[0, 0].set_ylabel("Predicted Ice Cover", fontsize=12)
axes[1, 0].set_ylabel("Target Ice Cover", fontsize=12)

plt.show()